In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model

print("TensorFlow version:", tf.__version__)


In [ ]:
input_texts = ["hi", "hello", "bye", "thanks", "yes", "no"]
target_texts = ["hola", "hola", "adios", "gracias", "si", "no"]

print("Input texts :", input_texts)
print("Target texts:", target_texts)


In [ ]:
target_texts = ["\t" + text + "\n" for text in target_texts]

print("Target texts with start/end symbols:")
for t in target_texts:
    print(repr(t))


In [ ]:
input_chars = sorted(list(set("".join(input_texts))))
target_chars = sorted(list(set("".join(target_texts))))

input_char_to_index = {char: i + 1 for i, char in enumerate(input_chars)}
target_char_to_index = {char: i + 1 for i, char in enumerate(target_chars)}

input_index_to_char = {i: char for char, i in input_char_to_index.items()}
target_index_to_char = {i: char for char, i in target_char_to_index.items()}

num_encoder_tokens = len(input_chars) + 1
num_decoder_tokens = len(target_chars) + 1

print("Input characters :", input_chars)
print("Target characters:", target_chars)
print("Number of encoder tokens:", num_encoder_tokens)
print("Number of decoder tokens:", num_decoder_tokens)


In [ ]:
max_encoder_seq_length = max(len(text) for text in input_texts)
max_decoder_seq_length = max(len(text) for text in target_texts)

print("Max encoder sequence length:", max_encoder_seq_length)
print("Max decoder sequence length:", max_decoder_seq_length)


In [ ]:
encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length), dtype="int32")
decoder_input_data = np.zeros((len(input_texts), max_decoder_seq_length), dtype="int32")
decoder_target_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    # Encoder input
    for t, char in enumerate(input_text):
        encoder_input_data[i, t] = input_char_to_index[char]

    # Decoder input and decoder target
    for t, char in enumerate(target_text):
        decoder_input_data[i, t] = target_char_to_index[char]
        if t > 0:
            decoder_target_data[i, t - 1, target_char_to_index[char]] = 1.0

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)


In [ ]:
latent_dim = 64

encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(input_dim=num_encoder_tokens, output_dim=16, mask_zero=True, name="encoder_embedding")(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]


In [ ]:
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_embedding_layer = Embedding(input_dim=num_decoder_tokens, output_dim=16, mask_zero=True, name="decoder_embedding")
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)


In [ ]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()


In [ ]:
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=300,
    verbose=1
)


In [ ]:
# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(latent_dim,), name="decoder_state_input_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_inf = decoder_embedding_layer(decoder_inputs)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding_inf, initial_state=decoder_states_inputs
)

decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)


In [ ]:
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1), dtype="int32")
    target_seq[0, 0] = target_char_to_index["\t"]

    decoded_sentence = ""

    while True:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = target_index_to_char.get(sampled_token_index, "")

        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            break

        decoded_sentence += sampled_char

        target_seq = np.zeros((1, 1), dtype="int32")
        target_seq[0, 0] = sampled_token_index

        states_value = [h, c]

    return decoded_sentence


In [ ]:
for seq_index in range(len(input_texts)):
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print(f"Input: {input_texts[seq_index]}  -->  Predicted Output: {decoded_sentence}")
